<a href="https://colab.research.google.com/github/Saumya7verma/Machine-Learning/blob/main/Spam_Email_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn nltk

In [2]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### **Download NLTK Stopwords**

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### **Load Dataset**

In [4]:
# Load Dataset
df = pd.read_csv('spam.csv', encoding='latin-1')

### **Keep Required Columns Only**

In [5]:
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### **Create Text Cleaning Function**

In [6]:
def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])

    # Remove stopwords
    words = text.split()

    words = [word for word in words if word not in stopwords.words('english')]

    return " ".join(words)

### **Apply Cleaning**

In [7]:
df['clean_message'] = df['message'].apply(clean_text)

df.head()

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


### **Convert Text into TF-IDF Features**

In [8]:
tfidf = TfidfVectorizer(ngram_range=(1,2))

X = tfidf.fit_transform(df['clean_message'])

y = df['label']

### **Split Dataset**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### **Train Naive Bayes Model**

In [10]:
model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

### **Make Predictions**

In [11]:
y_pred = model.predict(X_test)

Check Accuracy

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9282511210762332


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.93      1.00      0.96       965
        spam       0.97      0.48      0.64       150

    accuracy                           0.93      1115
   macro avg       0.95      0.74      0.80      1115
weighted avg       0.93      0.93      0.92      1115



### **Test Your Own Messages**

In [14]:
message = ["URGENT! You have won a FREE cash prize. Claim now"]

cleaned_message = [clean_text(msg) for msg in message]

message_tfidf = tfidf.transform(cleaned_message)

prediction = model.predict(message_tfidf)

print(prediction)

['spam']


In [19]:
message = "Congratulations! You won a free lottery ticket claim now"

cleaned = clean_text(message)

vector = tfidf.transform([cleaned])

print(model.predict(vector))

print(model.predict_proba(vector))

['spam']
[[0.25989879 0.74010121]]


In [16]:
tests = [
    "URGENT! Claim your FREE cash prize now",
    "WINNER!! Call now to receive your reward",
    "Free entry in 2 a weekly competition to win tickets",
    "You have been selected for a prize claim now",
    "Hey bro are you coming to class today?"
]

for msg in tests:

    cleaned = clean_text(msg)

    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)

    print(msg, " ---> ", prediction[0])

URGENT! Claim your FREE cash prize now  --->  spam
WINNER!! Call now to receive your reward  --->  spam
Free entry in 2 a weekly competition to win tickets  --->  spam
You have been selected for a prize claim now  --->  spam
Hey bro are you coming to class today?  --->  ham


In [17]:
print("lottery" in tfidf.vocabulary_)
print("congratulations" in tfidf.vocabulary_)

False
True
